In [2]:
from pathlib import Path ## to use the Path class for file handling
from google import genai ## to use the Gemini API client
from datetime import datetime ## to get the current date and time
import json ## to handle JSON data
import pandas as pd ## to handle data in DataFrame format
import os ## to handle environment variables
import dtale

In [ ]:
# Get current local date and time
now = datetime.now()

##extract the date in YYYY-MM-DD format
today = now.strftime("%Y-%m-%d")
print("\nToday's date: " + today)


In [ ]:
#configure the paths for the notes and prompt template files
notes_path = Path("data/import/sample_notes.md")
prompt_path = Path("data/import/extraction_prompt_template.md")

In [ ]:
#storing the contents of the notes and prompt template files in variables
raw_notes = notes_path.read_text(encoding="utf-8")
prompt_template = prompt_path.read_text(encoding="utf-8")

print("Notes:")
print(raw_notes)

print("\nPrompt template:")
print(prompt_template)

In [ ]:
#configure the path for the Gemini API key file

gemini_api_key_path = Path("data/API_tokens_values/gemini_api_key.txt")

if not gemini_api_key_path.exists():
    raise FileNotFoundError(f"Gemini API key file not found: {gemini_api_key_path}")

with open(gemini_api_key_path, "r", encoding="utf-8") as f:
    gemini_api_key = f.read().strip()

#initialize the Gemini API client with the API key
client = genai.Client(api_key=gemini_api_key)

In [ ]:
#calling the Gemini API to generate content based on the prompt template and raw notes

#replace placeholders in the prompt template with actual values
prompt = (
    prompt_template
    .replace("{today_date}", today)
    .replace("{timezone}", "Asia/Kolkata")
    .replace("{raw_notes}", raw_notes)
)

response = client.models.generate_content(
    model="gemini-3.5-flash",
    contents=prompt,
)

print(response.text)

In [ ]:
#storing the response from the Gemini API in a JSON file

response_text = response.text.strip()

# If Gemini returns pure JSON text
data = json.loads(response_text)

output_path = Path("data/export/response.json")

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(data, f, indent=2, ensure_ascii=False)

print(f"Saved response to {output_path}")

In [3]:
with open("data/export/response.json", "r", encoding="utf-8") as f:
    tasks = json.load(f)

df = pd.DataFrame(tasks)

schema_columns = [
    "title",
    "event_type",
    "anchor_datetime",
    "estimated_duration",
    "lock_status",
]

df = df.reindex(columns=schema_columns)

df

Traceback (most recent call last):
  File "C:\Users\hemug\AppData\Local\Temp\ipykernel_7728\3449067815.py", line 502, in _repr_dw_
    return self._gen_json()
           ~~~~~~~~~~~~~~^^
  File "C:\Users\hemug\AppData\Local\Temp\ipykernel_7728\3449067815.py", line 510, in _gen_json
    return api["pandas_transport"]["get_df_payload"](tmp_vars[self.id]["converted"], max_num_rows_to_preload, **kwargs)
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\hemug\AppData\Local\Temp\ipykernel_7728\3449067815.py", line 250, in get_df_payload
    return get_df_payload_v0(df, row_limit, **extra_props)
  File "C:\Users\hemug\AppData\Local\Temp\ipykernel_7728\3449067815.py", line 242, in get_df_payload_v0
    return pd_dumps({
        **extra_props,
    ...<2 lines>...
        "columns": column_info_list
    })
  File "C:\Users\hemug\AppData\Local\Temp\ipykernel_7728\3449067815.py", line 219, in pd_dumps
    return

,title,event_type,anchor_datetime,estimated_duration,lock_status
0,ACE promotional layout meet,fixed_time,2026-08-16T18:00,NaN,locked
1,ML pipeline debug - YOLO,fixed_time,2026-08-17T20:00,NaN,movable
2,Calisthenics routine,fixed_time,2026-07-19T00:00,NaN,movable
3,Shoot YouTube banter vid,deadline_task,2026-07-18T23:59,NaN,movable
4,Ask papa about schedule,fixed_time,2026-07-17T00:00,NaN,movable
5,Flute practice,deadline_task,NaN,30.0,movable
6,Finish CCUS report,deadline_task,2026-07-18T23:59,NaN,locked


In [4]:
# Open D-Tale editor
d = dtale.show(
    df,
    name="Scheduler task review",
    allow_cell_edits=True,
)

d.open_browser()

In [5]:
edited_df = d.data

# 1. Convert float column to nullable integer, keeping nulls as NaN
edited_df['estimated_duration'] = edited_df['estimated_duration'].astype('Int64')


edited_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 5 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   title               7 non-null      str  
 1   event_type          7 non-null      str  
 2   anchor_datetime     6 non-null      str  
 3   estimated_duration  2 non-null      Int64
 4   lock_status         7 non-null      str  
dtypes: Int64(1), str(4)
memory usage: 419.0 bytes


In [6]:
edited_df.head()

Traceback (most recent call last):
  File "C:\Users\hemug\AppData\Local\Temp\ipykernel_7728\3449067815.py", line 502, in _repr_dw_
    return self._gen_json()
           ~~~~~~~~~~~~~~^^
  File "C:\Users\hemug\AppData\Local\Temp\ipykernel_7728\3449067815.py", line 510, in _gen_json
    return api["pandas_transport"]["get_df_payload"](tmp_vars[self.id]["converted"], max_num_rows_to_preload, **kwargs)
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\hemug\AppData\Local\Temp\ipykernel_7728\3449067815.py", line 250, in get_df_payload
    return get_df_payload_v0(df, row_limit, **extra_props)
  File "C:\Users\hemug\AppData\Local\Temp\ipykernel_7728\3449067815.py", line 242, in get_df_payload_v0
    return pd_dumps({
        **extra_props,
    ...<2 lines>...
        "columns": column_info_list
    })
  File "C:\Users\hemug\AppData\Local\Temp\ipykernel_7728\3449067815.py", line 219, in pd_dumps
    return

,title,event_type,anchor_datetime,estimated_duration,lock_status
0,ACE promotional layout meet,fixed_time,2026-08-16T18:00,<NA>,locked
1,ML pipeline debug - YOLO,fixed_time,2026-08-17T20:00,<NA>,movable
2,Calisthenics routine,fixed_time,2026-07-19T00:00,30,movable
3,Shoot YouTube banter vid,deadline_task,2026-07-18T23:59,<NA>,movable
4,Ask papa about schedule,fixed_time,2026-07-17T00:00,<NA>,movable


In [11]:
#export the edited DataFrame to a CSV file
output_csv_path = Path("data/export/CSV_2.csv")
edited_df.to_csv(output_csv_path, index=False)